# Coinbase candle backfill and inspection

This notebook uses Coinbase Advanced Trade's public one-minute candle API. It shares the same backfill function and SQLite database as `crypto-candles`, is safe to rerun, and never places trades.

In [1]:
from pathlib import Path

import pandas as pd

from crypto_trader.data.coinbase_candles import (
    backfill_historical_candles,
    open_candle_database,
)

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
database_path = project_root / "data" / "database" / "trading.db"
products = ["BTC-USD", "ETH-USD", "SOL-USD", "XRP-USD", "DOGE-USD"]
backfill_days = 30
connection = open_candle_database(database_path)

## Backfill

Change `products` or `backfill_days` above, then run this cell. Coinbase limits each response to 350 candles, so the function walks through the requested range in consecutive pages. Coinbase occasionally omits a historical minute bucket; those omissions appear as warnings, but the backfill continues and does not invent replacement prices.

In [2]:
processed_rows = backfill_historical_candles(
    connection,
    products=products,
    minutes=backfill_days * 24 * 60,
)
print(f"Processed {processed_rows:,} candle rows.")

Coinbase omitted 1 one-minute candle(s) for BTC-USD; first missing open_time_ms=1790959620000. Keeping the gap rather than inventing OHLCV data.
Coinbase omitted 1 one-minute candle(s) for ETH-USD; first missing open_time_ms=1790959620000. Keeping the gap rather than inventing OHLCV data.
Coinbase omitted 1 one-minute candle(s) for SOL-USD; first missing open_time_ms=1790959620000. Keeping the gap rather than inventing OHLCV data.
Coinbase omitted 1 one-minute candle(s) for XRP-USD; first missing open_time_ms=1789198080000. Keeping the gap rather than inventing OHLCV data.
Coinbase omitted 1 one-minute candle(s) for XRP-USD; first missing open_time_ms=1790959620000. Keeping the gap rather than inventing OHLCV data.
Coinbase omitted 6 one-minute candle(s) for DOGE-USD; first missing open_time_ms=1788995280000. Keeping the gap rather than inventing OHLCV data.
Coinbase omitted 36 one-minute candle(s) for DOGE-USD; first missing open_time_ms=1789008540000. Keeping the gap rather than inve

Processed 214,333 candle rows.


## Inspect coverage and schema

These queries show the time coverage for each product, any missing source minutes, recent prices, and the SQLite schema.

In [3]:
observed_times = pd.read_sql_query(
    """
    SELECT symbol, open_time_ms
    FROM candles
    WHERE exchange = 'coinbase' AND interval_seconds = 60
    ORDER BY symbol, open_time_ms
    """,
    connection,
)

gap_rows = []
for symbol, rows in observed_times.groupby("symbol"):
    observed = pd.Index(rows["open_time_ms"])
    expected = pd.RangeIndex(observed.min(), observed.max() + 60_000, 60_000)
    for open_time_ms in expected.difference(observed):
        gap_rows.append({"symbol": symbol, "open_time_ms": open_time_ms})

source_gaps = pd.DataFrame(gap_rows, columns=["symbol", "open_time_ms"])
if not source_gaps.empty:
    source_gaps["open_time_utc"] = pd.to_datetime(
        source_gaps["open_time_ms"], unit="ms", utc=True
    )
source_gaps


,symbol,open_time_ms,open_time_utc
0,BTC-USD,1790959620000,2026-10-02 16:47:00+00:00
1,DOGE-USD,1788995280000,2026-09-09 23:08:00+00:00
2,DOGE-USD,1788996960000,2026-09-09 23:36:00+00:00
3,DOGE-USD,1788997080000,2026-09-09 23:38:00+00:00
4,DOGE-USD,1788997440000,2026-09-09 23:44:00+00:00
...,...,...,...
1662,DOGE-USD,1791575760000,2026-10-09 19:56:00+00:00
1663,ETH-USD,1790959620000,2026-10-02 16:47:00+00:00
1664,SOL-USD,1790959620000,2026-10-02 16:47:00+00:00
1665,XRP-USD,1789198080000,2026-09-12 07:28:00+00:00


In [4]:
coverage = pd.read_sql_query(
    """
    SELECT exchange, symbol, interval_seconds, COUNT(*) AS rows,
           MIN(open_time_ms) AS first_open_time_ms,
           MAX(open_time_ms) AS last_open_time_ms
    FROM candles
    WHERE exchange = 'coinbase'
    GROUP BY exchange, symbol, interval_seconds
    ORDER BY symbol
    """,
    connection,
)
coverage

,exchange,symbol,interval_seconds,rows,first_open_time_ms,last_open_time_ms
0,coinbase,BTC-USD,60,43203,1788984120000,1791576300000
1,coinbase,DOGE-USD,60,41538,1788984360000,1791576300000
2,coinbase,ETH-USD,60,43203,1788984120000,1791576300000
3,coinbase,SOL-USD,60,43203,1788984120000,1791576300000
4,coinbase,XRP-USD,60,43202,1788984120000,1791576300000


In [5]:
latest = pd.read_sql_query(
    """
    SELECT symbol, datetime(open_time_ms / 1000, 'unixepoch') AS open_time_utc,
           open, high, low, close, volume
    FROM candles
    WHERE exchange = 'coinbase' AND interval_seconds = 60
    ORDER BY open_time_ms DESC, symbol
    LIMIT 25
    """,
    connection,
)
latest

,symbol,open_time_utc,open,high,low,close,volume
0,BTC-USD,2026-10-09 20:05:00,82251.32000,82268.75000,82235.79000,82263.91000,3.100770
1,DOGE-USD,2026-10-09 20:05:00,0.08447,0.08450,0.08444,0.08450,15924.200000
2,ETH-USD,2026-10-09 20:05:00,2474.42000,2474.94000,2473.27000,2474.69000,149.806470
3,SOL-USD,2026-10-09 20:05:00,108.55000,108.61000,108.45000,108.60000,1403.191307
4,XRP-USD,2026-10-09 20:05:00,1.38630,1.38680,1.38560,1.38670,11361.041690
5,BTC-USD,2026-10-09 20:04:00,82254.58000,82254.58000,82240.24000,82251.31000,3.606502
6,DOGE-USD,2026-10-09 20:04:00,0.08448,0.08451,0.08446,0.08451,27226.700000
7,ETH-USD,2026-10-09 20:04:00,2475.21000,2475.21000,2473.60000,2474.43000,126.950481
8,SOL-USD,2026-10-09 20:04:00,108.60000,108.60000,108.53000,108.55000,196.120065
9,XRP-USD,2026-10-09 20:04:00,1.38640,1.38670,1.38560,1.38640,64061.337858


In [6]:
schema = pd.read_sql_query(
    "SELECT type, name, sql FROM sqlite_master WHERE sql IS NOT NULL ORDER BY type, name",
    connection,
)
schema

,type,name,sql
0,index,idx_candles_interval_open_time,CREATE INDEX idx_candles_interval_open_time\nO...
1,table,candles,CREATE TABLE candles (\n -- Internal row id...


In [7]:
connection.close()